# 🚀 Быстрый старт: Использование NLP SDK

Этот ноутбук демонстрирует, как использовать класс `NLPPipeline` для запуска модели классификации в несколько строк кода. Фасад автоматически подтягивает конфигурацию, настраивает устройство (CPU/GPU) и загружает веса.

> ⚠️ **Важно:** без явно указанных обученных весов (`checkpoint_path`, `hf_repo_id` или `use_mlflow_registry: True` в конфиге) классификатор строится на **базовой предобученной модели** (например, `DeepPavlov/rubert-base-cased`) со **случайно инициализированной головой классификации**. Предсказания в этом случае бессмысленны (близки к 50/50) — это нормально и ожидаемо. Этот ноутбук демонстрирует **интерфейс и способ работы SDK**, а не качество классификации.

Ниже показаны варианты загрузки реальных обученных весов, если они у вас есть.

In [1]:
# Настройка логирования для красивого вывода в ноутбуке
import logging
import sys
from pathlib import Path


logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Настраиваем пути, чтобы ноутбук видел модули из папки src
project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

## Инициализация классификатора

При инициализации SDK:
1. Автоматически парсится `configs/main.yaml`
2. Загружается токенизатор и базовая модель (`HFModelBuilder`)
3. Переносится на GPU (если доступно)

Есть три варианта, откуда взять веса:

- **Ничего не указывать** — используется то, что прописано в `configs/model/default.yaml` (по умолчанию — базовая непредобученная под задачу модель, см. предупреждение выше).
- **`use_mlflow_registry: True`** в конфиге модели — веса подтягиваются из вашего MLflow Model Registry по алиасу (например, `Production`). Требует доступа к вашему MLflow tracking server.
- **`hf_repo_id` + `checkpoint_path`** — если вы (или ваша команда) выложили обученный чекпоинт на Hugging Face Hub.

In [2]:
from src.sdk.inference import NLPPipeline


# Вариант 1 (по умолчанию, используется ниже): базовая модель из конфига,
# без обученной головы — интерфейс работает, но предсказания случайны.
classifier = NLPPipeline()

# Вариант 2: если у вас есть доступ к вашему MLflow tracking server и
# модель зарегистрирована с алиасом Production — включите это в
# configs/model/default.yaml (use_mlflow_registry: True) и просто
# вызовите NLPPipeline() как обычно, builder сам подтянет веса оттуда.

# Вариант 3: если обученный чекпоинт выложен на HF Hub:
# classifier = NLPPipeline(hf_repo_id="your_name/fake-news-model", checkpoint_path="best.ckpt")

# Тестовый прогон
result = classifier("Власти опровергли слухи о закрытии станций метро.")
print(result)

c:\fake-news-detection-ml-system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: Инициализация NLPPipeline (Classification) на устройстве: cpu
INFO: Загрузка токенизатора: DeepPavlov/rubert-base-cased
INFO: 🔍 Запрос модели 'models:/FakeNewsDetector@Production' из MLflow Model Registry...
2026/07/21 00:06:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/21 00:06:19 INFO mlflow.store.db.utils: Updating database tables
INFO: Загрузка модели из: DeepPavlov/rubert-base-cased
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18549.79it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------

[{'label_id': 0, 'confidence': 0.555, 'all_probabilities': [0.555, 0.445]}]


## Запуск Инференса

Мы можем передавать как одиночные строки, так и батчи (списки строк). Метод `predict` возвращает структурированный словарь с вероятностями и предсказанным классом.

In [3]:
# Тестовые данные (имитация входящих новостей)
sample_messages = [
    "Здравствуйте, подскажите статус моей заявки №12345?",
    "ШОК! Инопланетяне высадились в Саратове! Смотреть без смс...",
    "Центробанк сохранил ключевую ставку на уровне 16%."
]

results = classifier(sample_messages)

for msg, res in zip(sample_messages, results, strict=True):
    print("-" * 50)
    print(f"Текст: {msg}")
    print(f"Класс (ID): {res['label_id']}")
    print(f"Уверенность: {res['confidence'] * 100:.2f}%")

print("\n(Если веса не были обученным чекпоинтом — уверенность ожидаемо близка к 50%, см. предупреждение в начале ноутбука.)")

--------------------------------------------------
Текст: Здравствуйте, подскажите статус моей заявки №12345?
Класс (ID): 0
Уверенность: 54.02%
--------------------------------------------------
Текст: ШОК! Инопланетяне высадились в Саратове! Смотреть без смс...
Класс (ID): 0
Уверенность: 52.33%
--------------------------------------------------
Текст: Центробанк сохранил ключевую ставку на уровне 16%.
Класс (ID): 0
Уверенность: 59.67%

(Если веса не были обученным чекпоинтом — уверенность ожидаемо близка к 50%, см. предупреждение в начале ноутбука.)
